## Description

## Import libraries

In [1]:
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client.models import Filter, FieldCondition, MatchValue, Prefetch, SparseVector, FusionQuery
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient
import cohere
from dotenv import load_dotenv
import os
from google import genai
import math

In [2]:
load_dotenv()

True

## RAG pipeline implementation

In [3]:
def get_dense_embedding(dense_model, text):

    return next(dense_model.embed([text])).tolist()

In [4]:
def get_sparse_embedding(sparse_model, text):

    sparse_embedding = next(sparse_model.embed([text]))

    values = sparse_embedding.values.tolist()
    indices = sparse_embedding.indices.tolist()

    return values, indices

In [5]:
def retrieval(query, qdrant_client, dense_model, sparse_model, relevant_years, limit_vector_search=200, limit_keywords_search=200, 
                                                                                                                        limit_result=100):

    dense_embedding = get_dense_embedding(dense_model, query)
    sparse_embedding_values, sparse_embedding_indices = get_sparse_embedding(sparse_model, query)


    query_filter = None

    if relevant_years != []:

        query_filter = Filter(
            should=[
                FieldCondition(
                    key="year",
                    match=MatchValue(value=year)
                )
                for year in relevant_years
            ]
        )


    results = qdrant_client.query_points(
        collection_name="regulations",
        with_payload=True,
        prefetch=[
            Prefetch(
                query=dense_embedding,
                using="dense",
                limit=limit_vector_search,
                filter=query_filter
            ),
            Prefetch(
                query=SparseVector(
                    indices=sparse_embedding_indices,
                    values=sparse_embedding_values
                ),
                using="sparse",
                limit=limit_keywords_search,
                filter=query_filter
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=limit_result,
    )


    return results.points

In [6]:
def reranking(query, list_points, cohere_client, top_n):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)


    response = cohere_client.rerank(
        model="rerank-v4.0-pro",
        query=query,
        documents=texts,
        top_n=top_n,
    )
    
    reranking_result = [ list_points[item.index] for item in response.results ]


    return reranking_result

In [14]:
def generate_multi_queries(query, gemini_client, number_generated_queries=4):

    gemini_prompt = f"""
    Generate {number_generated_queries} alternative search queries for the following question.

    The queries should have the same meaning but use different wording.

    Question:
    {query}

    Return only the queries, one per line.
    """

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=gemini_prompt
    )

    return [query] + response.text.split("\n")

In [8]:
def reciprocal_rank_fusion(result_lists, k=60):

    id_point = {}

    for points_list in result_lists:

        for idx, point in enumerate(points_list):

            id = point.id

            if id not in id_point:
                id_point[id] = point
                id_point[id].score = 0

            id_point[id].score += ( 1 / (k + (idx+1)) )


    rrf_result = sorted( list(id_point.items()), key=lambda x: x[1].score, reverse=True )
    rrf_result = list( dict(rrf_result).values() )


    return rrf_result     

In [22]:
def generate_answer(user_query, list_points, gemini_client):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)

    context = '\n\n'.join(texts)


    prompt = f"""
    You are an expert on FIA Formula 1 Regulations.

    Answer the question using ONLY the provided context.

    Rules:

    - Do not use outside knowledge.
    - If the answer is not available in the context, say:
    "The provided regulations do not contain enough information to answer this question."
    - For every factual statement, provide the corresponding year, regulation type, and article number.
    - When comparing regulations from different years, clearly indicate which year and article each statement comes from.
    - If multiple articles are relevant, cite all of them.
    - Be precise and concise.
    - Use bullet points when comparing regulations.
    - When answering comparison questions, focus only on substantive regulatory changes.
    - When comparing regulations, focus on the content and meaning of the rules rather than article numbering.
    - Do NOT assume that a rule has changed solely because its article number has changed.
    - Do NOT treat article renumbering as a regulatory change.
    - Do NOT treat changes in article references as a regulatory change unless the regulatory meaning has changed.
    - Do NOT treat formatting or wording differences as regulatory changes unless they affect the meaning of the rule.
    - Before reporting a change, verify that the actual rule content differs between the years.
    - If the provided context does not contain enough information to determine whether a rule has changed, explicitly state this.

    Context:
    {context}

    Question:
    {user_query}
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )


    return response.text


In [15]:
def extract_years(query, gemini_client):

    prompt = f"""
    Extract Formula 1 regulation years mentioned in the question.

    If a range is requested, return all years in that range.

    Return ONLY a LIST of integers.

    Examples:

    Question:
    What were the Safety Car rules in 2021?

    Answer:
    [2021]

    Question:
    Compare the 2018 and 2026 regulations.

    Answer:
    [2018, 2026]

    Question:
    How did the regulations change from 2020 to 2026?

    Answer:
    [2020, 2021, 2022, 2023, 2024, 2025, 2026]

    Question:
    How did the regulations evolve between 2019 and 2022?

    Answer:
    [2019, 2020, 2021, 2022]

    Question:
    In which year was the fastest lap point removed?

    Answer:
    []

    Question:
    {query}
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )


    return eval(response.text)

In [11]:
def rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client):

    relevant_years = extract_years(user_query, gemini_client)

    queries = generate_multi_queries(user_query, gemini_client)

    hybrid_search_results_lists = [ retrieval(query, qdrant_client, dense_model, sparse_model, relevant_years) for query in queries]
    
    rrf_result = reciprocal_rank_fusion(hybrid_search_results_lists)


    reranking_top_n = 20
    reranking_result = []

    if len(relevant_years) <= 1:
        reranking_result = reranking(user_query, rrf_result, cohere_client, reranking_top_n)
    else:
        top_n_per_year = math.ceil( reranking_top_n / len(relevant_years) ) 

        for year in relevant_years:

            necessary_points = list( filter( lambda x: x.payload['year'] == year, rrf_result ) )
            reranking_year_res = reranking(user_query, necessary_points, cohere_client, min(top_n_per_year, len(necessary_points)))

            reranking_result += reranking_year_res


    final_answer = generate_answer(user_query, reranking_result, gemini_client)


    return final_answer, reranking_result, rrf_result, relevant_years

## Testing retrieve and generation

In [12]:
dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/minicoil-v1"
)

qdrant_client = QdrantClient(path='../data/qdrant_storage')

cohere_client = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
# 'Please tell me about the Safety Car rules in 2025 year.' 
# 'How have the Safety Car rules changed between 2018 and 2026?' 
# 'Please tell me about the Safety Car rules in 2018 year.' 

In [23]:
user_query = "How have the Safety Car regulations changed in 2020 compared to 2019?"
answer, retrieved_chunks, rrf_result, relevant_years = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [24]:
print(relevant_years)

[2019, 2020]


In [26]:
qdrant_client.close()

In [25]:
print(answer)

The following changes have occurred in the Safety Car regulations in 2020 compared to 2019, based solely on the provided context:

*   **Removed Regulations:**
    *   The regulation detailing the safety car's pre-formation lap procedure in 2019 has been removed.
        *   2019 sporting regulation Article 39.2: "Fifty minutes before the start of the formation lap the safety car will leave the pit lane and take up position at the front of the grid and remain there until the five minute signal is given. At this point (except under Article 36.14(c)) it will cover a whole lap of the circuit and take up position."
    *   The regulation prohibiting unnecessarily slow, erratic, or dangerous driving while the safety car is deployed in 2019 has been removed.
        *   2019 sporting regulation Article 39.5: "No car may be driven unnecessarily slowly, erratically or in a manner which could be deemed potentially dangerous to other drivers or any other person at any time whilst the safety car 

In [140]:
print(rrf_result)

[ScoredPoint(id='78bce355-3749-4dcf-8f96-85ac31d46af3', version=0, score=0.08119877049180328, payload={'year': 2024, 'regulation_type': 'sporting', 'article': '55.10', 'chapter': 'SAFETY CAR', 'content': 'Except under Article 55.123 below, the safety car shall be used at least until the leader is behind it and all remaining cars are lined up behind him. Once behind the safety car, the leader must keep within ten (10) car lengths of it (except under Article 55.134 below).'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='bcff580e-1470-4b45-82dc-ee0e9ea01c45', version=0, score=0.07838827838827839, payload={'year': 2024, 'regulation_type': 'sporting', 'article': '55.6', 'chapter': 'SAFETY CAR', 'content': 'The safety car will join the track with its orange lights illuminated and will do so regardless of where the leader is.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='4b428c3e-6d6c-4776-9263-996dd2255cc8', version=0, score=0.07795266841968816, payloa

In [19]:
for i, point in enumerate(retrieved_chunks, start=1):                #retrieved_chunks, start=1):
    p = point.payload

    print("=" * 120)
    print(f"Result #{i}")
    print(f"Score      : {point.score:.4f}")
    print(f"Year       : {p['year']}")
    print(f"Article    : {p['article']}")
    print(f"Chapter    : {p['chapter']}")
    print("-" * 120)
    print(p['content'])
    print()

Result #1
Score      : 0.0707
Year       : 2019
Article    : 39.10
Chapter    : SAFETY CAR
------------------------------------------------------------------------------------------------------------------------
Except under 39.12 below, the safety car shall be used at least until the leader is behind it and all remaining cars are lined up behind him. Once behind the safety car, the race leader must keep within ten car lengths of it (except under 39.13 below).

Result #2
Score      : 0.0530
Year       : 2019
Article    : 39.3
Chapter    : SAFETY CAR
------------------------------------------------------------------------------------------------------------------------
The safety car may be brought into operation to neutralise a race upon the order of the clerk of the course. It will be used only if competitors or officials are in immediate physical danger on or near the track but the circumstances are not such as to necessitate suspending the race.

Result #3
Score      : 0.0516
Year  

Сформувати 10 питань і оцінити на них знайдені чанки та якість фінальної відповіді